## Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

## Read bronze table

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2")

## Silver transformation

In [0]:
%sql
select *
from workspace.bronze.erp_px_cat_g1v2
LIMIT 10


ID,CAT,SUBCAT,MAINTENANCE
AC_BR,Accessories,Bike Racks,Yes
AC_BS,Accessories,Bike Stands,No
AC_BC,Accessories,Bottles and Cages,No
AC_CL,Accessories,Cleaners,Yes
AC_FE,Accessories,Fenders,No
AC_HE,Accessories,Helmets,Yes
AC_HP,Accessories,Hydration Packs,No
AC_LI,Accessories,Lights,Yes
AC_LO,Accessories,Locks,Yes
AC_PA,Accessories,Panniers,No


### Trimming


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Normalize maintenance flag to boolean

In [0]:
df = (
    df.withColumn(
        "MAINTENANCE",
        F.when(F.col("MAINTENANCE") == "Yes", F.lit(True))
        .when(F.col("MAINTENANCE") == "No", F.lit(False))
        .otherwise(None)
    )
)

### Rename columns

In [0]:
RENAME_MAP = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE": "maintenance_flag",
}

for old, new in RENAME_MAP.items():
    df = df.withColumnRenamed(old, new)

### Sanity check of dataframe

In [0]:
df.limit(20).display()

category_id,category,subcategory,maintenance_flag
AC_BR,Accessories,Bike Racks,true
AC_BS,Accessories,Bike Stands,false
AC_BC,Accessories,Bottles and Cages,false
AC_CL,Accessories,Cleaners,true
AC_FE,Accessories,Fenders,false
AC_HE,Accessories,Helmets,true
AC_HP,Accessories,Hydration Packs,false
AC_LI,Accessories,Lights,true
AC_LO,Accessories,Locks,true
AC_PA,Accessories,Panniers,false


### Write silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_product_category")

### Sanity check of silver table

In [0]:
%sql
SELECT *
FROM workspace.silver.erp_product_category
LIMIT 20

category_id,category,subcategory,maintenance_flag
AC_BR,Accessories,Bike Racks,true
AC_BS,Accessories,Bike Stands,false
AC_BC,Accessories,Bottles and Cages,false
AC_CL,Accessories,Cleaners,true
AC_FE,Accessories,Fenders,false
AC_HE,Accessories,Helmets,true
AC_HP,Accessories,Hydration Packs,false
AC_LI,Accessories,Lights,true
AC_LO,Accessories,Locks,true
AC_PA,Accessories,Panniers,false
